In [ ]:
from IPython.display import display
from typing import TypedDict, Annotated, Literal

from dotenv import  load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph,START,END
from loguru import logger

load_dotenv(override=True)

model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 声明状态
#1.1 全局状态
class OverAllState(TypedDict):
    username:str
    user_input:str
    output:str
#1.2 私有状态
class StructuredOutputState(TypedDict):
    topic:Annotated[str, ..., "主题"]
    mode:Annotated[Literal["poem","joke"], ..., "poem 表示写诗，joke 表示写笑话"]

model_with_structure=model.with_structured_output(schema=StructuredOutputState)


def router_node(state:OverAllState)->StructuredOutputState:
    logger.info("路由节点已经执行")
    user_input=state["user_input"]
    res=model_with_structure.invoke([HumanMessage(user_input)])
    logger.info("路由结果为:{}",res)
    return res
#2. 声明节点
def router(state:StructuredOutputState)->Literal["node_poem","node_joke","node_default"]:
    logger.info("路由函数已经执行")
    if state["mode"]=="poem":
        logger.info("路由至node_poem节点")
        return "node_poem"
    elif state["mode"]=="joke":
        logger.info("路由至node_joke节点")
        return "node_joke"
    logger.info("路由至兜底节点")
    return "node_default"
# 2.1 写诗节点
def node_poem(state:StructuredOutputState)->OverAllState:
    logger.info("node_poem正在执行")
    topic=state["topic"]
    poem=model.invoke([
        HumanMessage(f"写一首关于{topic}主题的七言绝句,不要赏析,只写诗句的部分")
    ]).content
    return{
        "output":poem
    }

# 2.2 写笑话节点
def node_joke(state:StructuredOutputState)->OverAllState:
    logger.info("node_joke正在执行")
    topic=state["topic"]
    joke=model.invoke([
        HumanMessage(f"写一首关于{topic}主题的笑话,字数在100以内")
    ]).content
    return{
        "output":joke
    }
#2.3 兜底节点
def node_default(state:StructuredOutputState) -> OverAllState:
    logger.info(f"node_default已经执行")
    return {
        "output":"无法处理的任务类型"
    }
#3. 构件图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("router_node",router_node)
builder.add_node("node_poem",node_poem)
builder.add_node("node_joke",node_joke)
builder.add_node("node_default",node_default)

builder.add_edge(START,"router_node")
builder.add_conditional_edges("router_node",router,path_map = ["node_poem","node_joke","node_default"])
builder.add_edge("node_poem",END)
builder.add_edge("node_joke",END)
builder.add_edge("node_default",END)

#4. 添加检查点后端

checkpointer=InMemorySaver()
config={"configurable":{"thread_id":"chapter03-08"}}
graph=builder.compile(checkpointer=checkpointer)
display(graph)
res=graph.invoke({
    "username":"张三",
    "user_input":"写一首关于荷花的诗"
},config=config)
print(res)


In [ ]:
history_checkpoints = list(graph.get_state_history(config=config))
print(history_checkpoints)

In [ ]:
# 1. 获取某一个节点的位置
before_router_checkpoint = next(h for h in history_checkpoints if h.next == ('router_node',))
print(before_router_checkpoint)

In [ ]:
change_input = graph.update_state(
    config=before_router_checkpoint.config,
    values={"user_input":"帮我写一个荷花的笑话"},
    as_node=START
)
print(change_input)
graph.get_state(change_input)

In [ ]:
res = graph.invoke(None,config=change_input)
print(res)

In [ ]:
# 将给定状态更新视为 router_node 的输出，观察返回配置中的后续路由并继续执行
skip_router_config = graph.update_state(
    config=before_router_checkpoint.config,
    values={"topic":"狸花猫","mode":"joke"},
    as_node="router_node"
)

print(skip_router_config)

graph.get_state(skip_router_config)

In [ ]:
graph.invoke(None,config=skip_router_config)